In [24]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from imblearn.pipeline import Pipeline as ImbPipeline


from src.features.new_features import TimeFeatureStrategy, AgeCategoryFeatureStrategy, \
    HourCategoryFeatureStrategy, NewFeature

from imblearn.over_sampling import SMOTE


In [2]:
df_transactions = pd.read_csv('../data/raw/transactions.csv', encoding='utf-8')
df_customers = pd.read_csv('../data/raw/customers.csv', encoding='utf-8')
df = pd.merge(
    df_transactions,
    df_customers,
    left_on='sender_id',
    right_on='customer_id',
    how='left'
)

In [3]:
features = [
    # DateFeatureStrategy(),
    TimeFeatureStrategy(),
    AgeCategoryFeatureStrategy(),
    HourCategoryFeatureStrategy()
]

for f in features:
    processor = NewFeature(f)
    df_features = processor.apply(df)

In [4]:
df_features.head()

,transaction_id,timestamp,sender_id,receiver_id,amount,device_type,fraud,customer_id,cpf,age,gender,pix_key,account_type,city,hour_date,minute_date,age_category,hour_category
0,1,2025-07-09 23:11:19,51,525,705.05,mobile,0,51,069.753.842-74,61,F,sajoao-lucas@example.net,corrente,Ferreira de Santos,23,11,idoso,madrugada
1,2,2025-04-15 05:19:52,783,581,480.89,mobile,0,783,423.596.701-07,29,M,sramos@example.org,corrente,Lima da Prata,5,19,jovem,manha
2,3,2025-05-09 03:03:49,324,937,2198.60,app,0,324,164.850.329-24,23,M,lucas89@example.net,corrente,Ramos,3,3,jovem,madrugada
3,4,2025-07-18 12:06:22,789,845,2219.27,mobile,0,789,453.876.190-75,24,F,brendaalmeida@example.net,corrente,Sales da Mata,12,6,jovem,tarde
4,5,2025-05-08 00:42:09,141,769,4627.92,app,0,141,910.342.857-50,28,M,scamara@example.net,poupança,Pinto,0,42,jovem,madrugada


In [5]:
df_features.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 18 columns):
 #   Column          Non-Null Count  Dtype   
---  ------          --------------  -----   
 0   transaction_id  15000 non-null  int64   
 1   timestamp       15000 non-null  object  
 2   sender_id       15000 non-null  int64   
 3   receiver_id     15000 non-null  int64   
 4   amount          15000 non-null  float64 
 5   device_type     15000 non-null  object  
 6   fraud           15000 non-null  int64   
 7   customer_id     15000 non-null  int64   
 8   cpf             15000 non-null  object  
 9   age             15000 non-null  int64   
 10  gender          15000 non-null  object  
 11  pix_key         15000 non-null  object  
 12  account_type    15000 non-null  object  
 13  city            15000 non-null  object  
 14  hour_date       15000 non-null  int32   
 15  minute_date     15000 non-null  int32   
 16  age_category    15000 non-null  category
 17  hour_categor

In [6]:
df_features.head()

,transaction_id,timestamp,sender_id,receiver_id,amount,device_type,fraud,customer_id,cpf,age,gender,pix_key,account_type,city,hour_date,minute_date,age_category,hour_category
0,1,2025-07-09 23:11:19,51,525,705.05,mobile,0,51,069.753.842-74,61,F,sajoao-lucas@example.net,corrente,Ferreira de Santos,23,11,idoso,madrugada
1,2,2025-04-15 05:19:52,783,581,480.89,mobile,0,783,423.596.701-07,29,M,sramos@example.org,corrente,Lima da Prata,5,19,jovem,manha
2,3,2025-05-09 03:03:49,324,937,2198.60,app,0,324,164.850.329-24,23,M,lucas89@example.net,corrente,Ramos,3,3,jovem,madrugada
3,4,2025-07-18 12:06:22,789,845,2219.27,mobile,0,789,453.876.190-75,24,F,brendaalmeida@example.net,corrente,Sales da Mata,12,6,jovem,tarde
4,5,2025-05-08 00:42:09,141,769,4627.92,app,0,141,910.342.857-50,28,M,scamara@example.net,poupança,Pinto,0,42,jovem,madrugada


In [25]:
list_onehot = ['hour_category', 'age_category', 'account_type', 'gender', 'device_type']
list_num = ['amount', 'age']

cat_pipeline = Pipeline([
    ('onehot', OneHotEncoder(handle_unknown='ignore',sparse_output=False)),
])
num_pipeline = Pipeline([
    ('num', StandardScaler()),
])

preprocessor = ColumnTransformer([
    ('num', num_pipeline, list_num),
    ('cat', cat_pipeline, list_onehot),
])

pipeline = ImbPipeline([
    ('preprocessor', preprocessor),
    ('smote', SMOTE()),
    ('classifier', LogisticRegression())
])

pipeline.fit(df_features, df_features['fraud'])



,steps,"[('preprocessor', ...), ('smote', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


---